### Set-up

In [26]:
## setting paths and all
from pathlib import Path  ### switching to pathlib path-handling instead of the os - should work consistently between HPC and local machine
import sys

# current working directory of the notebook / script
cwd = Path.cwd()

# assume "scripts" folder is one level up from cwd
project_root = cwd.parent.resolve()

if str(project_root) not in sys.path:
    print("Adding to sys.path:", project_root)
    sys.path.append(str(project_root)) # add root to Python path (as a string) for finding scripts modules further


#2
# 0. loading libraries (could be removed after all modules are loaded from the scripts)
import glob
import tifffile
from bioio import BioImage
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage.color import rgb2gray
from skimage.color import label2rgb

from skimage.exposure import rescale_intensity
from skimage.transform import resize

from cellpose import models
from stardist.models import StarDist3D 

# loading scripts
from scripts.pre_processing import preprocess_3d_image ### pre-processing function (normalisation + optional downsampling)
from scripts.segment_3d import segment_with_stardist, segment_cytoplasm ### StarDist3D (3d_demo) segmentation model - light and relatively quick to run
from scripts.io_utils import load_multichannel_images, save_segmentation_results


ImportError: cannot import name 'segment_cytoplasm' from 'scripts.segment_3d' (C:\Users\Lada\Desktop\job\tsn_rocks\Image-Analysis-Summer-Project\scripts\segment_3d.py)

In [22]:
###### configuration ######

#input_folder = project_root.parent.resolve() / "input_data" / "Images_BBBC050" / "test" / "Images" # put the directory to your input data, relative to the project folder
                                                                                                   # project_root.parent() derives directory above the project folder root
#input_folder = project_root.parent.resolve() / "input_data" / "BBBC035" / "BBBC035_v1_dataset" / "01" 

input_folder = project_root.parent.resolve() / "input_data" / "2017_07_21_Tom20" / "AICS-11-part13" 

#input_folder = project_root.parent.resolve() / "input_data" / "S-BIAD1272_30min_stimulation" ### .lif files


In [23]:
# 1. load images as multichannel dictionary

channel_map = {
    "nucleus": 2,
    "cytoplasm": 0,
    "mitochondria": 1
}

all_volumes = load_multichannel_images(input_folder, channel_map)

# preview loaded images: first image [0] as example
print(all_volumes[0]["filename"])       # e.g., "sample01.tif"
print(all_volumes[0]["channels"].keys()) # dict_keys(['nucleus', 'cytoplasm', 'mitochondria'])
print(all_volumes[0]["channels"]["nucleus"].shape)  # (Z, Y, X)

AICS-11_1301.ome
dict_keys(['nucleus', 'cytoplasm', 'mitochondria'])
(60, 624, 924)


#### MAIN: per-channel preprocessing, segmentation, and quantification

In [24]:
### configs ###
#OUTPUT_DIR = project_root.parent / "output_data" / "Images_BBBC050" / "test" / "Images" 
OUTPUT_DIR = project_root.parent / "output_data" / "2017_07_21_Tom20" / "AICS-11-part13" 
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### params ###
num_test_volumes = 10  # number of volumes from `frames` to segment
use_model = "stardist"  # "cellpose" or "stardist"
diameter = None        # or specify an estimate, e.g., 20
channels = [0, 0]      # grayscale input for Cellpose

### pre-processing params ###
downsize_factor = 0.5   # scaling factor or 1 to keep original                                 TODO: make sure that downsampling is proportionate? (e.g. if the original size is not 1:1)
per_slice_norm = False        # True = normalize per-slice, False = normalize whole stack


##### a) nucleus

In [25]:
# 2. nucleus segmentation

results_dict = {}

# loop over loaded images for segmentation
for v in all_volumes:
    file_name = v['filename']  # preserve original file name
    
    # --- nuclei segmentation ---
    nuclei_volume = v["channels"]["nucleus"]
    nuclei_norm = preprocess_3d_image(nuclei_volume, downsize_factor=downsize_factor, per_slice=per_slice_norm)
    
    if use_model == "stardist":
        nuclei_mask = segment_with_stardist(nuclei_norm)
    elif use_model == "cellpose":
        nuclei_mask = segment_with_cellpose(nuclei_norm, diameter=diameter, channels=channels)
    
    # saving masks in a dictionary for future use
    if file_name not in results_dict:
        results_dict[file_name] = {}    # initialize dictionary for this file if it doesn't exist

    results_dict[file_name]["nuclei_mask"] = nuclei_mask 
    
    # save nuclei mask
    save_segmentation_results(
        nuclei_norm, 
        nuclei_mask, 
        output_root=OUTPUT_DIR, 
        experiment_label=f"{file_name}_nuclei",
        save_overlay=True
    )


Running StarDist...
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.707933, nms_thresh=0.3.
[INFO] Saved mask stack: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\2017_07_21_Tom20\AICS-11-part13\AICS-11_1301\mask.tif
[INFO] Saved MIP overlay: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\2017_07_21_Tom20\AICS-11-part13\AICS-11_1301\overlay_MIP.png
Running StarDist...
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.707933, nms_thresh=0.3.
[INFO] Saved mask stack: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\2017_07_21_Tom20\AICS-11-part13\AICS-11_1302\mask.tif
[INFO] Saved MIP overlay: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\2017_07_21_Tom20\AICS-11-part13\AICS-11_1302\overlay_MIP.png
Running StarDist...
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: p

In [ ]:
# (optional): visualise the example nucleus segmentation result using Napari (have not checked on myriad)

%gui qt
import napari

viewer = napari.Viewer(ndisplay=3)
viewer.add_image(nuclei_norm, name='Raw Image', rendering='mip', colormap='gray') ### adding the original 3d grey-scale image
viewer.add_labels(nuclei_mask, name='Segmentation Mask') ### adding a created mask overlay

##### b) cytoplasm

In [27]:
# 3. cytoplasm segmentation

# loop over loaded images for segmentation
for v in all_volumes:
    file_name = v['filename']  # preserve original file name

    # get the corresponding nuclear mask
    if file_name not in results_dict:
        raise KeyError(f"Nuclear mask for {file_name} not found in previous results.")
    nuclei_mask = results_dict[file_name]["nuclei_mask"]

    # --- cytoplasm segmentation (watershed) ---
    cytoplasm_volume = v["channels"]['cytoplasm']
    cytoplasm_norm = preprocess_3d_image(
            cytoplasm_volume, 
            downsize_factor=downsize_factor, 
            per_slice=per_slice_norm
        )    
    cytoplasm_mask = segment_cytoplasm(
        nuclei_mask,
        cytoplasm_norm, 
        mode="membrane",  # could also be "cytoskeleton" for alternative approach
        membrane_threshold=0.1
    )
    
    # saving masks in a dictionary for future use
    if file_name not in results_dict:
        results_dict[file_name] = {}    # initialize dictionary for this file if it doesn't exist
    
    results_dict[file_name]["cytoplasm_mask"] = cytoplasm_mask


    save_segmentation_results(
        cytoplasm_norm, 
        cytoplasm_mask, 
        output_root=OUTPUT_DIR, 
        experiment_label=f"{file_name}_cytoplasm",
        save_overlay=True
    )

[INFO] Saved mask stack: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\2017_07_21_Tom20\AICS-11-part13\AICS-11_1301\mask.tif
[INFO] Saved MIP overlay: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\2017_07_21_Tom20\AICS-11-part13\AICS-11_1301\overlay_MIP.png
[INFO] Saved mask stack: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\2017_07_21_Tom20\AICS-11-part13\AICS-11_1302\mask.tif
[INFO] Saved MIP overlay: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\2017_07_21_Tom20\AICS-11-part13\AICS-11_1302\overlay_MIP.png
[INFO] Saved mask stack: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\2017_07_21_Tom20\AICS-11-part13\AICS-11_1303\mask.tif
[INFO] Saved MIP overlay: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\2017_07_21_Tom20\AICS-11-part13\AICS-11_1303\overlay_MIP.png
[INFO] Saved mask stack: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\2017_07_21_Tom20\AICS-11-part13\AICS-11_1304\mask.tif
[INFO] Saved MIP overlay: C:\Users\Lada\Desktop\job\tsn_rocks\output_data\2017_07_21_To

In [20]:
# (optional): visualise the example cytoplasm segmentation result using Napari (have not checked on myriad)

%gui qt
import napari

viewer = napari.Viewer(ndisplay=3)
viewer.add_image(cytoplasm_norm, name='Raw Image', rendering='mip', colormap='gray') ### adding the original 3d grey-scale image
viewer.add_labels(cytoplasm_mask, name='Segmentation Mask') ### adding a created mask overlay
viewer.add_labels(nuclei_mask, name='Nuc Segmentation Mask') ### adding a created mask overlay

<Labels layer 'Nuc Segmentation Mask' at 0x164ea0ad180>

##### c) organelle